In [1]:
from pathlib import Path
import pickle
import json

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


In [3]:
ROOT_DIR = Path("..").resolve()
DATA_DIR = ROOT_DIR / "data" / "EC3D"

with open(DATA_DIR / "ec3d_sequences.pkl", "rb") as f:
    ec3d_data = pickle.load(f)

with open(DATA_DIR / "split_cross_subject.json", "r") as f:
    split = json.load(f)

sequences = ec3d_data["sequences"]
labels = ec3d_data["labels"]
meta = ec3d_data["meta"]

train_indices = np.array(split["train_indices"], dtype=int)
test_indices = np.array(split["test_indices"], dtype=int)

print("N sequences:", len(sequences))
print("Train:", len(train_indices), "Test:", len(test_indices))
print("Example meta:", meta[0])


N sequences: 371
Train: 283 Test: 88
Example meta: {'sequence_key': 'Lunges_Hugues_1_1', 'exercise': 'Lunges', 'subject': 'Hugues', 'instruction_id': 1, 'instruction_name': 'Correct', 'global_label_id': 6, 'global_label_name': 'Lunges - Correct', 'trial_id': 1, 'num_frames': 64}


In [4]:
def extract_features_from_sequence(seq: np.ndarray) -> np.ndarray:
    # seq: (T, 3, 25)
    if seq.ndim != 3 or seq.shape[1] != 3:
        raise ValueError(f"Seq shape inattesa: {seq.shape}")
    seq_t = np.transpose(seq, (0, 2, 1))   # (T, 25, 3)
    root = seq_t[:, 0:1, :]               # (T, 1, 3)
    seq_centered = seq_t - root

    mean = seq_centered.mean(axis=0)
    std = seq_centered.std(axis=0)
    min_ = seq_centered.min(axis=0)
    max_ = seq_centered.max(axis=0)
    rng = max_ - min_

    feats = np.concatenate(
        [mean.flatten(), std.flatten(), rng.flatten()],
        axis=0
    )
    return feats.astype(np.float32)


In [5]:
X = []
y = []

for seq, lab in zip(sequences, labels):
    X.append(extract_features_from_sequence(seq))
    y.append(int(lab))

X = np.stack(X, axis=0)        # (N, 225)
y = np.array(y, dtype=np.int64)

print("X shape:", X.shape)
print("y shape:", y.shape, "classi:", np.unique(y))


X shape: (371, 225)
y shape: (371,) classi: [ 0  1  2  3  4  5  6  7  8  9 10 11]


In [7]:
ID_TO_FEEDBACK_EN = {
    0: "Your squat looks correct: neutral spine, knees tracking over toes and good depth.",
    1: "Your stance is too wide. Bring your feet slightly closer so they are about shoulder-width apart.",
    2: "Your knees are caving inward. Gently push them outward to keep them aligned with your feet.",
    3: "You are not squatting low enough. Try to sit a bit deeper while keeping your back neutral.",
    4: "You are bending your torso too far forward. Keep your chest up and your spine more upright.",
    5: "Your squat form is inconsistent. Focus on a stable stance and controlled movement.",
    6: "Your lunge form looks correct: stable step, good depth and front knee over the ankle.",
    7: "Your lunge is too shallow. Drop your back knee closer to the floor to increase the range of motion.",
    8: "Your front knee goes past your toes. Shorten your step or sit more downward instead of forward.",
    9: "Your plank looks correct: straight line from head to heels and engaged core.",
    10: "Your hips are too high in the plank. Lower them slightly to keep a straight line from shoulders to ankles.",
    11: "Your hips are dropping in the plank. Lift them a bit to avoid arching your lower back.",
}
NUM_CLASSES = len(ID_TO_FEEDBACK_EN)


In [8]:
class EC3DPoseTextDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = X[indices]
        self.y = y[indices]

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        pose_feat = self.X[idx]
        label = self.y[idx]       # 0..11
        return torch.from_numpy(pose_feat), torch.tensor(label, dtype=torch.long)
    

train_dataset = EC3DPoseTextDataset(X, y, train_indices)
test_dataset = EC3DPoseTextDataset(X, y, test_indices)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

len(train_dataset), len(test_dataset)


(283, 88)

In [9]:
EMBED_DIM = 128

class PoseEncoder(nn.Module):
    def __init__(self, in_dim=225, embed_dim=EMBED_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Linear(256, embed_dim),
        )

    def forward(self, x):
        # x: (B, 225)
        return self.net(x)

class TextEncoder(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, embed_dim=EMBED_DIM):
        super().__init__()
        self.embedding = nn.Embedding(num_classes, embed_dim)

    def forward(self, labels):
        # labels: (B,) int
        return self.embedding(labels)

class PoseTextCLIP(nn.Module):
    def __init__(self, in_dim=225, embed_dim=EMBED_DIM, num_classes=NUM_CLASSES):
        super().__init__()
        self.pose_encoder = PoseEncoder(in_dim, embed_dim)
        self.text_encoder = TextEncoder(num_classes, embed_dim)
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def forward(self, pose_feats, labels):
        pose_emb = self.pose_encoder(pose_feats)     # (B, D)
        text_emb = self.text_encoder(labels)         # (B, D)

        # normalizziamo per cosine similarity
        pose_emb = F.normalize(pose_emb, dim=-1)
        text_emb = F.normalize(text_emb, dim=-1)

        logit_scale = self.logit_scale.exp()
        logits = logit_scale * pose_emb @ text_emb.t()  # (B, B)

        return logits, pose_emb, text_emb


In [10]:
def clip_contrastive_loss(logits):
    """
    logits: (B, B) = sim(pose_i, text_j)
    target: i deve matchare j (diagonale)
    """
    B = logits.size(0)
    target = torch.arange(B, device=logits.device)
    loss_i2t = F.cross_entropy(logits, target)
    loss_t2i = F.cross_entropy(logits.t(), target)
    return (loss_i2t + loss_t2i) / 2


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = PoseTextCLIP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


Using device: cpu


In [12]:
from tqdm.auto import tqdm

def eval_retrieval(dataloader, model):
    model.eval()
    all_pose = []
    all_text = []
    all_labels = []

    with torch.no_grad():
        for pose_feats, labels in dataloader:
            pose_feats = pose_feats.to(device).float()
            labels = labels.to(device)
            _, pose_emb, text_emb = model(pose_feats, labels)
            all_pose.append(pose_emb)
            all_text.append(text_emb)
            all_labels.append(labels)

    pose_emb = F.normalize(torch.cat(all_pose, dim=0), dim=-1)
    text_emb = F.normalize(torch.cat(all_text, dim=0), dim=-1)
    labels = torch.cat(all_labels, dim=0)

    sims = pose_emb @ text_emb.t()
    preds = sims.argmax(dim=1)
    acc = (preds == torch.arange(len(labels), device=device)).float().mean().item()
    return acc

n_epochs = 20
for epoch in range(1, n_epochs+1):
    model.train()
    running_loss = 0.0
    for pose_feats, labels in train_loader:
        pose_feats = pose_feats.to(device).float()
        labels = labels.to(device)

        logits, _, _ = model(pose_feats, labels)
        loss = clip_contrastive_loss(logits)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * pose_feats.size(0)

    train_loss = running_loss / len(train_dataset)
    test_acc = eval_retrieval(test_loader, model)

    print(f"Epoch {epoch:02d} | train loss: {train_loss:.4f} | test retrieval acc (diag): {test_acc:.3f}")


Epoch 01 | train loss: 2.9869 | test retrieval acc (diag): 0.045
Epoch 02 | train loss: 2.1955 | test retrieval acc (diag): 0.080
Epoch 03 | train loss: 1.8859 | test retrieval acc (diag): 0.068
Epoch 04 | train loss: 1.7340 | test retrieval acc (diag): 0.091
Epoch 05 | train loss: 1.6424 | test retrieval acc (diag): 0.091
Epoch 06 | train loss: 1.5180 | test retrieval acc (diag): 0.068
Epoch 07 | train loss: 1.5465 | test retrieval acc (diag): 0.068
Epoch 08 | train loss: 1.4446 | test retrieval acc (diag): 0.080
Epoch 09 | train loss: 1.5074 | test retrieval acc (diag): 0.091
Epoch 10 | train loss: 1.4388 | test retrieval acc (diag): 0.080
Epoch 11 | train loss: 1.3972 | test retrieval acc (diag): 0.091
Epoch 12 | train loss: 1.3252 | test retrieval acc (diag): 0.091
Epoch 13 | train loss: 1.3452 | test retrieval acc (diag): 0.091
Epoch 14 | train loss: 1.2992 | test retrieval acc (diag): 0.091
Epoch 15 | train loss: 1.2865 | test retrieval acc (diag): 0.080
Epoch 16 | train loss: 1.